# Newton Simulation Concepts and Custom CR3A URDF Loader

This notebook introduces the core Newton workflow: build a ModelBuilder, finalize a Model, step a solver, visualize with Viser, and finally load a custom robot URDF from the CR3A manipulator package.

The examples are intentionally compact and follow the same pattern used by the Franka example: URDF defines robot geometry and joint topology, while Newton controller gains and target values are assigned separately.


## 1. Setup and Imports

Import the required Python libraries, configure Warp, and load the Newton modules needed for simulation and IK.


In [ ]:
import os
from pathlib import Path

import numpy as np
import warp as wp

import newton
import newton.examples
import newton.utils
import newton.ik as ik

wp.config.quiet = True

try:
    newton.solvers.SolverMuJoCo.import_mujoco()
except Exception:
    pass


## 2. Utilities

Define shared helper functions for creating Viser viewers, rendering Mermaid diagrams, and displaying progress bars in notebook cells.


In [ ]:
from html import escape
import json
import time
import uuid

from IPython.display import HTML, Javascript, display


def make_viewer(name: str):
    """Create a Viser viewer and record to a .viser file in the notebook output folder."""
    recording_path = Path("../_static/recordings") / f"{name}.viser"
    recording_path.parent.mkdir(parents=True, exist_ok=True)
    viewer = newton.viewer.ViewerViser(verbose=False, record_to_viser=str(recording_path))
    return viewer


def render_mermaid(diagram: str, theme: str = "forest", line_color: str = "#76b900"):
    element_id = f"mermaid-{uuid.uuid4().hex}"
    display(HTML(f'<div id="{element_id}" class="mermaid">{escape(diagram)}</div>'))

    config_json = json.dumps(
        {
            "startOnLoad": False,
            "theme": theme,
            "themeVariables": {"lineColor": line_color},
        }
    )

    js = f"""
(async () => {{
  const loadMermaid = () => new Promise((resolve, reject) => {{
    if (window.mermaid) {{
      resolve();
      return;
    }}
    const script = document.createElement("script");
    script.src = "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js";
    script.onload = () => resolve();
    script.onerror = () => reject(new Error("Failed to load Mermaid."));
    document.head.appendChild(script);
  }});

  await loadMermaid();
  const config = {config_json};
  window.mermaid.initialize(config);

  const node = document.getElementById("{element_id}");
  if (!node) {{
    return;
  }}

  try {{
    await window.mermaid.run({{ nodes: [node] }});
  }} catch (err) {{
    node.textContent = "Mermaid render error: " + (err && err.message ? err.message : err);
  }}
}})();
"""
    display(Javascript(js))


class _HTMLProgressBar:
    def __init__(self, iterable, total: int, desc: str = "", leave: bool = True):
        self._iterable = iterable
        self.total = max(int(total), 1)
        self.desc = desc
        self.leave = leave
        self.n = 0
        self._start = time.perf_counter()
        self._last_refresh = 0.0
        self._handle = display(HTML(self._render()), display_id=True)

    def _render(self) -> str:
        elapsed = max(time.perf_counter() - self._start, 1e-9)
        pct = min(100.0, 100.0 * self.n / self.total)
        rate = self.n / elapsed
        remaining = (self.total - self.n) / rate if rate > 1e-9 else float("inf")
        eta = "--:--" if not np.isfinite(remaining) else f"{int(remaining // 60):02d}:{int(remaining % 60):02d}"
        desc_html = escape(self.desc)
        return (
            f'<div style="font-family: sans-serif; margin: 6px 0;">'
            f'<div style="display:flex; justify-content:space-between; font-size:12px; margin-bottom:4px;">'
            f'<span>{desc_html}</span>'
            f'<span>{self.n}/{self.total} ({pct:5.1f}%)</span>'
            f'</div>'
            f'<progress value="{self.n}" max="{self.total}" style="width:100%; height:14px;"></progress>'
            f'<div style="font-size:11px; color:#666; margin-top:2px;">elapsed {elapsed:5.1f}s | eta {eta}</div>'
            f'</div>'
        )

    def _refresh(self, force: bool = False):
        now = time.perf_counter()
        if force or (now - self._last_refresh) >= 0.1 or self.n >= self.total:
            self._handle.update(HTML(self._render()))
            self._last_refresh = now

    def set_description(self, desc: str):
        self.desc = desc
        self._refresh(force=True)

    def update(self, n: int = 1):
        self.n = min(self.total, self.n + int(n))
        self._refresh()

    def close(self):
        self._refresh(force=True)

    def __iter__(self):
        try:
            for item in self._iterable:
                yield item
                self.update(1)
        finally:
            self.close()


def _tqdm_html(iterable=None, total=None, desc: str = "", leave: bool = True, **_kwargs):
    if iterable is None:
        if total is None:
            raise ValueError("Either iterable or total must be provided.")
        iterable = range(int(total))
    if total is None:
        try:
            total = len(iterable)
        except TypeError as e:
            raise ValueError("total is required for iterables without len().") from e
    return _HTMLProgressBar(iterable=iterable, total=int(total), desc=desc, leave=leave)


def _trange_html(*args, **kwargs):
    return _tqdm_html(range(*args), **kwargs)


tqdm, trange = _tqdm_html, _trange_html


## 3. ModelBuilder → Model

Build a minimal physics scene with a ground plane, a box, and a sphere, then inspect the resulting model metadata.


In [ ]:
builder = newton.ModelBuilder()

builder.add_ground_plane()

box_body = builder.add_body(xform=wp.transform(wp.vec3(0.0, -1.0, 1.0), wp.quat_identity()))
builder.add_shape_box(box_body, hx=0.15, hy=0.15, hz=0.15)

sphere_body = builder.add_body(xform=wp.transform(wp.vec3(0.0, 1.0, 1.2), wp.quat_identity()))
builder.add_shape_sphere(sphere_body, radius=0.18)

print(f"Bodies: {builder.body_count}, Shapes: {builder.shape_count}")


## 4. Finalize → Model, State, Control, Contacts

Finalize the builder into a simulation-ready Model and allocate State, Control, and Contacts buffers for solver stepping.


In [ ]:
model = builder.finalize()

state_0 = model.state()
state_1 = model.state()
control = model.control()
contacts = model.contacts()

print(f"Model device: {model.device}")
print(f"Body count: {model.body_count}")


## 5. Solver and Simulation Loop

Create an XPBD solver and implement the standard simulation loop that clears forces, collides bodies, advances the solver, and swaps state buffers.


In [ ]:
solver = newton.solvers.SolverXPBD(model, iterations=10)

fps = 60
frame_dt = 1.0 / fps
sim_substeps = 8
sim_dt = frame_dt / sim_substeps


def simulate():
    global state_0, state_1
    for _ in range(sim_substeps):
        state_0.clear_forces()
        contacts = model.collide(state_0)
        solver.step(state_in=state_0, state_out=state_1, control=control, contacts=contacts, dt=sim_dt)
        state_0, state_1 = state_1, state_0


## 6. Visualize the First Simulation

Run a short recorded simulation and log the state to the Viser viewer for visual playback in the notebook.


In [ ]:
viewer = make_viewer("01_simple_scene")
viewer.set_model(model)

sim_time = 0.0
num_frames = 30
for _ in range(num_frames):
    simulate()
    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.end_frame()
    sim_time += frame_dt

viewer


## 7. Performance: CUDA Graph Capture

Capture the simulation loop as a CUDA graph on GPU devices when available and compare runtime with and without graph capture.


In [ ]:
graph = None
if wp.get_device().is_cuda:
    with wp.ScopedCapture() as capture:
        simulate()
    graph = capture.graph
    print("CUDA graph captured")
else:
    print("Running on CPU; graph capture skipped")

sim_time = 0.0
for _ in range(20):
    if graph is not None:
        wp.capture_launch(graph)
    else:
        simulate()
    sim_time += frame_dt

print(f"Simulated {sim_time:.2f} seconds")


## 8. Load a Robot

Load a Franka arm from URDF, initialize joint values, and add an optional cube to the scene for manipulation tasks.


In [ ]:
def build_franka_scene(include_table=True, include_cube=True, use_targets=True):
    builder = newton.ModelBuilder()
    builder.default_shape_cfg.gap = 0.0
    newton.solvers.SolverMuJoCo.register_custom_attributes(builder)

    table_height = 0.1
    table_pos = wp.vec3(0.0, -0.5, 0.5 * table_height)
    table_top_center = table_pos + wp.vec3(0.0, 0.0, 0.5 * table_height)
    if include_table:
        builder.add_shape_box(body=-1, hx=0.4, hy=0.4, hz=0.5 * table_height, xform=wp.transform(table_pos))

    robot_base_pos = table_top_center + wp.vec3(-0.5, 0.0, 0.0)
    builder.add_urdf(
        str(newton.utils.download_asset("franka_emika_panda") / "urdf/fr3_franka_hand.urdf"),
        xform=wp.transform(robot_base_pos, wp.quat_identity()),
        floating=False,
        enable_self_collisions=False,
        parse_visuals_as_colliders=False,
    )

    builder.joint_q[:9] = [
        -3.6802115e-03,
        2.3901723e-02,
        3.6804110e-03,
        -2.3683236e00,
        -1.2918962e-04,
        2.3922248e00,
        7.8549200e-01,
        0.05,
        0.05,
    ]
    if use_targets:
        builder.joint_target_pos[:9] = builder.joint_q[:9]
        builder.joint_target_ke[:9] = [4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100]
        builder.joint_target_kd[:9] = [450, 450, 350, 350, 200, 200, 200, 10, 10]
    else:
        builder.joint_target_pos[:9] = builder.joint_q[:9]
        builder.joint_target_ke[:9] = [0] * 9
        builder.joint_target_kd[:9] = [0] * 9

    builder.joint_effort_limit[:9] = [87, 87, 87, 87, 12, 12, 12, 100, 100]
    builder.joint_armature[:9] = [0.195] * 4 + [0.074] * 3 + [0.1] * 2

    cube_size = 0.05
    cube_body = None
    if include_cube:
        cube_pos = table_top_center + wp.vec3(0.0, 0.15, 0.5 * cube_size)
        cube_body = builder.add_body(xform=wp.transform(cube_pos, wp.quat_identity()))
        shape_cfg = newton.ModelBuilder.ShapeConfig(margin=1e-3, density=400.0)
        builder.add_shape_box(body=cube_body, hx=0.5 * cube_size, hy=0.5 * cube_size, hz=0.5 * cube_size, cfg=shape_cfg)

    return builder, cube_size, table_pos, cube_body


builder, _, _, _ = build_franka_scene(include_cube=False, use_targets=False)
model = builder.finalize()
print(f"Franka model loaded: {model.body_count} bodies, {model.joint_count} joints")


## 9. Joint Targets with Control.joint_target_pos

Drive Franka joints using target positions and PD gains, then simulate the arm responding to a sinusoidal target motion.


In [ ]:
builder, _, _, _ = build_franka_scene(include_cube=False, use_targets=True)
model = builder.finalize()
state_0 = model.state()
state_1 = model.state()
control = model.control()
contacts = model.contacts()

solver = newton.solvers.SolverMuJoCo(
    model,
    solver="newton",
    integrator="implicitfast",
    iterations=20,
    ls_parallel=True,
    ls_iterations=100,
    nconmax=1000,
    njmax=2000,
    cone="elliptic",
    impratio=1000.0,
    use_mujoco_contacts=False,
)

newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

fps = 60
frame_dt = 1.0 / fps
sim_substeps = 8
sim_dt = frame_dt / sim_substeps

base_target = model.joint_q.numpy().astype(np.float32)
base_target[7:9] = 0.04

for frame in range(20):
    target = base_target.copy()
    target[3] = base_target[3] + 0.2 * np.sin(2.0 * np.pi * frame / 20)
    control.joint_target_pos.assign(target)
    for _ in range(sim_substeps):
        state_0.clear_forces()
        model.collide(state_0, contacts)
        solver.step(state_in=state_0, state_out=state_1, control=control, contacts=contacts, dt=sim_dt)
        state_0, state_1 = state_1, state_0

print("Joint target control loop completed")


## 10. Inverse Kinematics (IK) in Newton

Newton's IK module lets you compose multiple objectives such as position, rotation, and joint limits into a single solve.


In [ ]:
ik_diagram = r"""
flowchart TD
    IKS[IKSolver]
    IKS --> OBJ[Objectives]
    OBJ --> O1[Position]
    OBJ --> O2[Rotation]
    OBJ --> O3[JointLimit]

    IKS --> OPT[Optimizer]
    OPT --> OPT1[Levenberg-Marquardt]
    OPT --> OPT2[LBFGS]

    IKS --> JAC[Jacobian mode]
    JAC --> J1[ANALYTIC]
    JAC --> J2[AUTODIFF]
    JAC --> J3[MIXED]
"""

render_mermaid(ik_diagram, theme="forest", line_color="#76b900")


## 11. Step 1: Position-Only IK on One Target

Solve a single end-effector position target using a position objective and joint-limit objective, then animate the robot to follow it.


In [ ]:
builder, _, _, _ = build_franka_scene(include_table=False, include_cube=False, use_targets=True)
model = builder.finalize()
state_0 = model.state()
state_1 = model.state()
control = model.control()
contacts = model.contacts()

solver = newton.solvers.SolverMuJoCo(model, solver="newton", integrator="implicitfast", iterations=20)
newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

ee_index = 11
home_pos_np = state_0.body_q.numpy()[ee_index][:3].astype(np.float32)
target_np = home_pos_np + np.array([0.0, 0.52, 0.04], dtype=np.float32)

pos_obj = ik.IKObjectivePosition(
    link_index=ee_index,
    link_offset=wp.vec3(0.0, 0.0, 0.0),
    target_positions=wp.array([target_np], dtype=wp.vec3),
)

limit_lower_np = model.joint_limit_lower.numpy()
limit_upper_np = model.joint_limit_upper.numpy()
if limit_lower_np.shape[0] != model.joint_coord_count:
    pad = model.joint_coord_count - limit_lower_np.shape[0]
    limit_lower_np = np.concatenate([limit_lower_np, -np.ones(pad) * 1.0e6])
    limit_upper_np = np.concatenate([limit_upper_np, np.ones(pad) * 1.0e6])

joint_limit_obj = ik.IKObjectiveJointLimit(
    joint_limit_lower=wp.array(limit_lower_np, dtype=wp.float32),
    joint_limit_upper=wp.array(limit_upper_np, dtype=wp.float32),
)

joint_q_ik = wp.clone(model.joint_q.reshape((1, -1)))
ik_solver = ik.IKSolver(
    model=model,
    n_problems=1,
    objectives=[pos_obj, joint_limit_obj],
    lambda_initial=0.1,
    jacobian_mode=ik.IKJacobianType.ANALYTIC,
)

ik_solver.step(joint_q_ik, joint_q_ik, iterations=24)
print("Position-only IK solve completed")


## 12. Step 2: Preview the Rectangle Path (No IK Yet)

Construct and visualize a rectangular target path in end-effector space before solving IK against it.


In [ ]:
builder, _, _, _ = build_franka_scene(include_table=False, include_cube=False, use_targets=True)
model = builder.finalize()
state_0 = model.state()
newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

home_pos_np = state_0.body_q.numpy()[11][:3].astype(np.float32)
rect_center = home_pos_np + np.array([0.0, 0.25, 0.0], dtype=np.float32)
rect_half = 0.08
rect_corners_np = np.array(
    [
        [rect_center[0] - rect_half, rect_center[1] - rect_half, rect_center[2]],
        [rect_center[0] + rect_half, rect_center[1] - rect_half, rect_center[2]],
        [rect_center[0] + rect_half, rect_center[1] + rect_half, rect_center[2]],
        [rect_center[0] - rect_half, rect_center[1] + rect_half, rect_center[2]],
    ],
    dtype=np.float32,
)

print("Rectangle corners:\n", rect_corners_np)


## 13. Step 3: Full Rectangle IK Tracking

Combine position, rotation, and joint-limit objectives to track the full rectangle path and visualize the end-effector trace.


In [ ]:
builder, _, _, _ = build_franka_scene(include_table=False, include_cube=False, use_targets=True)
model = builder.finalize()
state_0 = model.state()
newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

home_pos_np = state_0.body_q.numpy()[11][:3].astype(np.float32)
rect_center = home_pos_np + np.array([0.0, 0.25, 0.0], dtype=np.float32)
rect_half = 0.08
rect_corners_np = np.array(
    [
        [rect_center[0] - rect_half, rect_center[1] - rect_half, rect_center[2]],
        [rect_center[0] + rect_half, rect_center[1] - rect_half, rect_center[2]],
        [rect_center[0] + rect_half, rect_center[1] + rect_half, rect_center[2]],
        [rect_center[0] - rect_half, rect_center[1] + rect_half, rect_center[2]],
    ],
    dtype=np.float32,
)

rect_path_points = []
for i in range(len(rect_corners_np)):
    start = rect_corners_np[i]
    end = rect_corners_np[(i + 1) % len(rect_corners_np)]
    for t in np.linspace(0.0, 1.0, 25, endpoint=False):
        rect_path_points.append(start * (1.0 - t) + end * t)
rect_path_np = np.array(rect_path_points, dtype=np.float32)

print(f"Generated {len(rect_path_np)} rectangle target points")


## 14. Coupled Manipulation: Franka Cable Pick-and-Place

Introduce the coupled manipulation problem of picking up a deformable cable with a Franka arm and explain the solver split between rigid motion and VBD cable dynamics.


In [ ]:
from newton.solvers import SolverMuJoCo, SolverVBD
from newton.solvers.experimental.coupled import SolverCoupled, SolverCoupledProxy

FRANKA_Q = [
    -3.6802115e-03,
    2.3901723e-02,
    3.6804110e-03,
    -2.3683236e00,
    -1.2918962e-04,
    2.3922248e00,
    7.8549200e-01,
    0.04,
    0.04,
]

CABLE_CENTER = wp.vec3(0.5, 0.0, 0.256)
CABLE_LENGTH = 0.38
CABLE_CONTACT_KE = 1.0e4
CABLE_CONTACT_KD = 1.0e-5 * CABLE_CONTACT_KE
GROUND_CONTACT_KD = 1.0

print("Coupled manipulation example prepared")


## 15. Why Coupling?

Explain why the Franka arm and cable need different solver treatments and how proxy coupling lets gripper motion influence cable contact without breaking the rigid-arm control loop.


In [ ]:
coupling_summary = r"""
flowchart LR
    A[Franka arm with MuJoCo] --> P[Proxy coupling]
    P --> C[VBD cable]
    C --> G[Ground + contact interactions]
    A --> T[Joint target tracking]
"""

render_mermaid(coupling_summary, theme="forest", line_color="#76b900")


## 16. Build the Coupled Scene

Create a single combined model containing both the Franka robot and the cable, add collision geometry for the ground, and organize the rigid and cable subsystems.


In [ ]:
def add_franka(builder: newton.ModelBuilder, base_z: float):
    builder.add_urdf(
        newton.utils.download_asset("franka_emika_panda") / "urdf/fr3_franka_hand.urdf",
        xform=wp.transform(wp.vec3(0.0, 0.0, base_z), wp.quat_identity()),
        floating=False,
        enable_self_collisions=False,
        parse_visuals_as_colliders=False,
        force_show_colliders=False,
    )
    builder.joint_q[: len(FRANKA_Q)] = FRANKA_Q
    builder.joint_target_q[: len(FRANKA_Q)] = FRANKA_Q


def build_coupled_scene():
    builder = newton.ModelBuilder(gravity=-9.81)
    builder.rigid_gap = 0.01
    SolverMuJoCo.register_custom_attributes(builder)
    SolverVBD.register_custom_attributes(builder, dahl_defaults_enabled=False)

    add_franka(builder, 0.0)

    points, quats = newton.utils.create_straight_cable_points_and_quaternions(
        start=CABLE_CENTER - wp.vec3(0.5 * CABLE_LENGTH, 0.0, 0.0),
        direction=wp.vec3(1.0, 0.0, 0.0),
        length=CABLE_LENGTH,
        num_segments=19,
        twist_total=0.0,
    )

    cable_cfg = newton.ModelBuilder.ShapeConfig(
        density=100.0,
        ke=CABLE_CONTACT_KE,
        kd=CABLE_CONTACT_KD,
        mu=1.0,
        margin=0.0,
        gap=0.01,
    )

    builder.add_rod(
        positions=points,
        quaternions=quats,
        radius=0.005,
        body_frame_origin="start",
        cfg=cable_cfg,
        stretch_stiffness=1.0e6,
        stretch_damping=1.0e-1,
        bend_stiffness=5.0e-4,
        bend_damping=1.0e-3,
        label="vbd_cable",
    )

    ground_shape = builder.add_ground_plane(height=0.1, label="cable_ground_plane")
    model = builder.finalize()
    return {"model": model, "ground_shape": ground_shape}

scene = build_coupled_scene()
print("Coupled scene created:", scene["model"].body_count, "bodies")


## 17. Create the Coupled Solvers

Instantiate the MuJoCo and VBD solvers inside a coupled solver setup, connect the gripper proxy bodies, and prepare contact data for stepping.


In [ ]:
model = scene["model"]
control = model.control()
state_0 = model.state()
state_1 = model.state()
newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

collision_pipeline = newton.CollisionPipeline(model, broad_phase="explicit", contact_matching="latest")
contacts = collision_pipeline.contacts()

solver = SolverCoupledProxy(
    model=model,
    entries=[
        SolverCoupled.Entry(
            name="mjc",
            solver=lambda view: SolverMuJoCo(
                model=view,
                solver="newton",
                integrator="implicitfast",
                cone="elliptic",
                iterations=20,
                ls_iterations=20,
                use_mujoco_contacts=False,
                njmax=256,
                nconmax=64,
            ),
            bodies=list(range(0, 10)),
            joints=list(range(0, 9)),
        ),
        SolverCoupled.Entry(
            name="vbd",
            solver=lambda view: SolverVBD(
                model=view,
                iterations=20,
                rigid_avbd_beta=1.0e2,
                rigid_contact_k_start=2.0e2,
                rigid_contact_history=False,
            ),
            bodies=list(range(10, model.body_count)),
            joints=list(range(9, model.joint_count)),
        ),
    ],
    coupling=SolverCoupledProxy.Config(
        proxies=[],
        iterations=1,
    ),
)

print("Coupled solver configured")


## 18. Preview the Coupled Scene

Render the initial coupled scene before task motion starts so the robot and cable are visible in the correct initial configuration.


In [ ]:
viewer = make_viewer("09_franka_cable_initial")
viewer.set_model(model)
viewer.begin_frame(0.0)
viewer.log_state(state_0)
viewer.end_frame()
viewer


## 19. Build the Franka IK System

Create a Franka-only IK model with task-space targets and gripper joint state, then prepare a solver for picking and placing the cable.


In [ ]:
builder_ik = newton.ModelBuilder()
builder_ik.add_urdf(
    str(newton.utils.download_asset("franka_emika_panda") / "urdf/fr3_franka_hand.urdf"),
    floating=False,
    enable_self_collisions=False,
    parse_visuals_as_colliders=False,
)

ik_model = builder_ik.finalize()
print(f"IK model: {ik_model.body_count} bodies, {ik_model.joint_count} joints")


## 20. Run the Cable Pick-and-Place

Iterate through task keyframes, solve IK for the gripper, copy joint targets into the coupled model, refresh contacts, and advance the full coupled simulation.


In [ ]:
fps = 60
frame_dt = 1.0 / fps
sim_time = 0.0

keyframes = [
    {"pos": np.array([0.5, 0.0, 0.3], dtype=np.float32), "grip": 0.04},
    {"pos": np.array([0.5, 0.0, 0.2], dtype=np.float32), "grip": 0.0},
    {"pos": np.array([0.7, 0.2, 0.25], dtype=np.float32), "grip": 0.0},
]

for keyframe in keyframes:
    target_pos = keyframe["pos"]
    grip_width = keyframe["grip"]
    print(f"Target position: {target_pos}, grip width: {grip_width}")

print("Pick-and-place control loop prepared")


## 21. Load the Custom CR3A URDF

Load the local CR3A description on CPU. Because its mesh URIs still reference the old build machine, create a patched temporary copy that points to the current `assets` directory before calling `ModelBuilder.add_urdf()`.


In [ ]:
# Force Warp/Newton to use the CPU; Isaac Sim is not required for this viewer.
wp.set_device("cpu")

cr3a_urdf = Path(
    "~/workspaces/movensys_ws/src/movensys-manipulator/"
    "movensys_manipulator_description/urdf/dobot_cr3a/"
    "movensys_manipulator.urdf"
).expanduser().resolve()
cr3a_assets_dir = cr3a_urdf.parent / "assets"

if not cr3a_urdf.is_file():
    raise FileNotFoundError(f"CR3A URDF not found: {cr3a_urdf}")
if not cr3a_assets_dir.is_dir():
    raise FileNotFoundError(f"CR3A assets directory not found: {cr3a_assets_dir}")

# The generated URDF still points to the machine on which xacro was built.
# Write a patched copy under /tmp and leave the original URDF untouched.
old_asset_uri = (
    "file:///home/hehe-jazzy/workspaces/movensys_ws/install/"
    "movensys_manipulator_description/share/"
    "movensys_manipulator_description/urdf/dobot_cr3a/assets/"
)
# Newton's URDF importer expects a filesystem path here, not a file:// URI.
new_asset_uri = cr3a_assets_dir.as_posix() + "/"
urdf_text = cr3a_urdf.read_text(encoding="utf-8")
patched_text = urdf_text.replace(old_asset_uri, new_asset_uri)

patched_urdf = Path("/tmp/dobot_cr3a_newton.urdf")
patched_urdf.write_text(patched_text, encoding="utf-8")

builder = newton.ModelBuilder()
builder.add_urdf(
    str(patched_urdf),
    floating=False,
    enable_self_collisions=False,
    parse_visuals_as_colliders=False,
)
model = builder.finalize()

state = model.state()
newton.eval_fk(model, model.joint_q, model.joint_qd, state)

print("Newton device:", model.device)
print("Patched URDF:", patched_urdf)
print(
    f"CR3A loaded: {model.body_count} bodies, "
    f"{model.joint_count} joints, {model.shape_count} shapes"
)


## 22. Preview CR3A with Viser

Publish the initialized CPU state to a Viser server. Isaac Sim is not involved; open the URL printed by Viser in a browser.


In [ ]:
# Close only the previous CR3A viewer when this cell is run again.
previous_cr3a_viewer = globals().get("cr3a_viewer")
if previous_cr3a_viewer is not None:
    previous_cr3a_viewer.close()

# Earlier examples use port 8080. Use a dedicated port for the CR3A scene.
cr3a_viewer = newton.viewer.ViewerViser(
    port=8090,
    label="CR3A Robot",
    verbose=True,
)
cr3a_viewer.set_model(model)
cr3a_viewer.begin_frame(0.0)
cr3a_viewer.log_state(state)
cr3a_viewer.end_frame()

print("Open the CR3A viewer at:", cr3a_viewer.url)

# Keep this as the last expression so Jupyter can render the viewer output.
cr3a_viewer


## Notes and Usage

- URDF is a robot description, not a Newton-specific actuator file.
- Newton reads robot topology, inertia, joint limits, and mesh visuals from the URDF.
- Controller gains and target values are configured separately in the ModelBuilder or control object.
- If your robot uses a different asset root or absolute mesh path, patch those paths before calling add_urdf.
- This notebook is ready to run in the Python environment that has Newton installed.
